In [87]:
import torch

In [134]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

B, T, C = 5, 16, 32
E = 4  # num experts
K = 2  # top_k
x = torch.randn(B, T, C)

W_router = torch.randn(C, E)
W_expert_0 = torch.randn(C*4, C)

In [135]:
x_flat = x.reshape(-1, C)
print(f"x_flat.shape = {list(x_flat.shape)}")

logits = x_flat @ W_router
print(f"logits.shape = {list(logits.shape)}")

weights = torch.sigmoid(logits)
print(f"weights.shape = {list(weights.shape)}")

x_flat.shape = [80, 32]
logits.shape = [80, 4]
weights.shape = [80, 4]


In [136]:
values, indices = torch.topk(weights, 2, dim=-1)
print(f"values.shape = {list(values.shape)}")

values.shape = [80, 2]


In [141]:
outputs = torch.zeros((x_flat.size(0), C*4))
print(f"outputs.shape = {list(outputs.shape)}")

outputs.shape = [80, 128]


In [142]:
expert_0_selector = (indices == 0).any(dim=-1)
print(f"expert_0_selector.shape = {list(expert_0_selector.shape)}")

expert_0_selector.shape = [80]


In [143]:
x_flat_expert_0 = x_flat[expert_0_selector]
print(f"x_flat_expert_0.shape = {list(x_flat_expert_0.shape)}")

x_flat_expert_0.shape = [34, 32]


In [158]:
output_raw_flat_expert_0 = x_flat_expert_0 @ W_expert_0.t()
print(f"output_raw_flat_expert_0.shape = {list(output_raw_flat_expert_0.shape)}")

output_raw_flat_expert_0.shape = [34, 128]


In [159]:
expert_0_weights = weights[expert_0_selector, 0:1]
print(f"expert_0_weights.shape = {list(expert_0_weights.shape)}")

expert_0_weights.shape = [34, 1]


In [160]:
output_flat_expert_0 = output_raw_flat_expert_0 * expert_0_weights
print(f"output_flat_expert_0.shape = {list(output_flat_expert_0.shape)}")

output_flat_expert_0.shape = [34, 128]


In [161]:
outputs[expert_0_selector] += output_flat_expert_0